# Lesson 08 · RAG clássico · Demo em sala

Quatro células, uma por slide **LIVE**. Roda na máquina do professor com o Ollama local (`ollama serve`) e os modelos `qwen2.5:3b` e `nomic-embed-text` já baixados, ou no Google Colab com o bloco de preparação abaixo.

Toda célula imprime o próprio tempo (⏱) e a última célula resume os tempos da sessão, para planejar o ritmo da aula.

**Antes da aula** rode a célula 0 uma vez. Ela indexa o corpus no Chroma (uns 20 segundos) e persiste em disco, então as células 1 a 4 respondem em segundos durante a aula.

```
ollama pull qwen2.5:3b
ollama pull nomic-embed-text
pip install openai chromadb requests
```


---
## 🧰 Preparação no Colab (pule se estiver na sua máquina com o Ollama instalado)

Três células, uns 3 a 4 minutos. Rode antes de a aula começar e mantenha a aba ativa, porque a sessão do Colab expira com inatividade. Escolha o runtime **T4 GPU** para o gerador responder em segundos.

In [1]:
# Colab 1 · Instala o Ollama e sobe o servidor em segundo plano.
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2
import subprocess, time, requests
OLLAMA_URL = "http://localhost:11434"
def server_is_up():
    try: return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException: return False
if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up(): break
        time.sleep(1)
print("servidor no ar:", server_is_up())

>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
servidor no ar: True


In [2]:
# Colab 2 · Baixa os dois modelos da demo.
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:3b                 357c53fb659c    1.9 GB    5 seconds ago             


In [3]:
# Colab 3 · Pacotes Python.
!pip -q install openai chromadb requests pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

---
## Núcleo e cronômetro

In [4]:
# Núcleo do RAG clássico. As mesmas funções servem à demo em sala e ao notebook completo.
import os, re, glob, json, time, requests
from openai import OpenAI
import chromadb

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"            # o gerador (o Léo). Troque por "qwen2.5:0.5b" se a máquina sofrer.
EMB_MODEL  = "nomic-embed-text"      # o modelo de embedding. Tem de ser O MESMO na indexação e na consulta.
CORPUS_DIR = "corpus"
CHROMA_DIR = "chroma_lesson08"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ---- tokens. Aqui 1 token = 1 palavra (com o espaço que a precede), para o notebook não depender de nada.
# Tokenizadores reais (BPE) produzem cerca de 30% mais tokens que palavras. A proporção entre os tamanhos é o que importa.
def tokenize(text):   return re.findall(r"\s*\S+", text)
def detokenize(toks): return "".join(toks).strip()
def count_tokens(text): return len(tokenize(text))

# ---- Load
def load_corpus(folder=CORPUS_DIR):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*.md"))):
        docs.append({"doc": os.path.basename(path), "text": open(path, encoding="utf-8").read()})
    return docs

# ---- Chunk (tamanho fixo com overlap, em tokens)
def chunk_fixed(text, chunk_size=512, overlap=51):
    toks, step, out = tokenize(text), chunk_size - overlap, []
    for start in range(0, max(len(toks), 1), step):
        out.append(detokenize(toks[start:start + chunk_size]))
        if start + chunk_size >= len(toks):
            break
    return out

def make_chunks(docs, chunk_size=512, overlap=51):
    """Uma linha por chunk: id, texto e metadados (documento, posição, última seção vista)."""
    rows = []
    for d in docs:
        carried = "-"                                          # última seção vista no chunk anterior
        for i, c in enumerate(chunk_fixed(d["text"], chunk_size, overlap)):
            heads = [(m.start(), m.group(1)[:80]) for m in re.finditer(r"^#+ (.+)$", c, re.M)]
            first_half = [h for pos, h in heads if pos < len(c) / 2]
            section = first_half[-1] if first_half else carried   # a seção que domina o chunk
            if heads:
                carried = heads[-1][1]
            rows.append({"id": f"{d['doc']}#{i}", "text": c,
                         "doc": d["doc"], "chunk": i, "section": section})
    return rows

GROUNDING = ("You are the pit-wall assistant of Aurora Racing. Answer using ONLY the context below. "
             "If the context does not contain the answer, say exactly: \"The handbook does not cover this.\" "
             "Every fact you state must carry its source in square brackets, copied from the context labels, "
             "like [handbook_tyres.md · 2.2 Reference pressures]. "
             "End your answer with one line in exactly this format:\n"
             "Sources: [file · section], [file · section]")

def sources_used(hits):
    """A lista de fontes vem da recuperação, não do modelo. É o que se mostra ao usuário."""
    seen, out = set(), []
    for h in hits:
        key = f"{h['doc']} · {h['section']}"
        if key not in seen:
            seen.add(key); out.append(key)
    return out

# ---- Embed (Ollama /api/embed devolve uma lista de vetores)
def embed(texts, model=EMB_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": texts})
    r.raise_for_status()
    return r.json()["embeddings"]

# ---- Store (Chroma persistente, distância = 1 - cosseno)
def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)
    if rebuild:
        try: store.delete_collection(name)
        except Exception: pass
    col = store.get_or_create_collection(name, metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL})
    if col.count() == len(rows):
        return col                                   # já indexado; não paga o embedding de novo
    for i in range(0, len(rows), batch):
        part = rows[i:i + batch]
        col.upsert(ids=[r["id"] for r in part], documents=[r["text"] for r in part],
                   embeddings=embed([r["text"] for r in part]),
                   metadatas=[{"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]} for r in part])
    return col

# ---- Retrieve
def retrieve(col, question, k=4):
    assert col.metadata.get("embedding_model") == EMB_MODEL, "índice e consulta com modelos de embedding diferentes"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

# ---- Augment
GROUNDING = ("You are the pit-wall assistant of Aurora Racing. Answer using ONLY the context below. "
             "If the context does not contain the answer, say exactly: \"The handbook does not cover this.\" "
             "Cite the source of each fact in square brackets, like [handbook_tyres.md].")
FREE = "You are the pit-wall assistant of Aurora Racing. Answer the question."

def build_prompt(question, hits, grounded=True):
    context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
    system = GROUNDING if grounded else FREE
    user = f"Context:\n{context}\n\nQuestion: {question}" if hits else f"Question: {question}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

# ---- Generate
def generate(messages, model=GEN_MODEL, temperature=0.0, max_tokens=300):
    t0 = time.time()
    resp = client.chat.completions.create(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return resp.choices[0].message.content.strip(), time.time() - t0

def warm_up():
    """Carrega os dois modelos na memória e pede ao Ollama para mantê-los lá (keep_alive=-1). Evita pagar a carga na primeira pergunta."""
    requests.post(f"{OLLAMA_URL}/api/embed", json={"model": EMB_MODEL, "input": "warm up", "keep_alive": -1}).raise_for_status()
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": GEN_MODEL, "prompt": "hi", "keep_alive": -1, "stream": False,
                                                       "options": {"num_predict": 1}}).raise_for_status()
    for m in requests.get(f"{OLLAMA_URL}/api/ps").json().get("models", []):
        where = "GPU" if m.get("size_vram", 0) >= m.get("size", 1) * 0.9 else ("parcial GPU" if m.get("size_vram", 0) else "CPU")
        print(f"{m['name']:<24} carregado em {where}")

def show(hits):
    for i, h in enumerate(hits, 1):
        print(f"[{i}] {h['doc']} · {h['section']}  (distance {h['distance']:.3f})")
        print("    " + h["text"][:160].replace("\n", " ") + " ...")


In [5]:
# Cronômetro automático. A partir daqui toda célula imprime o seu tempo de execução (⏱) e o registra em CELL_TIMES.
import time
from IPython import get_ipython

CELL_TIMES = []
_ip = get_ipython()

def _pre_run(info):
    _ip._t0 = time.perf_counter()

def _post_run(result):
    dt = time.perf_counter() - getattr(_ip, "_t0", time.perf_counter())
    first_line = (result.info.raw_cell.strip().splitlines() or [""])[0][:70]
    CELL_TIMES.append({"cell": first_line, "seconds": round(dt, 2)})
    print(f"⏱ {dt:.1f} s")

if not getattr(_ip, "_timer_installed", False):          # evita registrar duas vezes se a célula for rerodada
    _ip._t0 = time.perf_counter()
    _ip.events.register("pre_run_cell", _pre_run)
    _ip.events.register("post_run_cell", _post_run)
    _ip._timer_installed = True
print("cronômetro ligado")


cronômetro ligado
⏱ 0.0 s


In [6]:
# O corpus vive dentro do notebook, para não depender de download. Esta célula grava os arquivos em ./corpus.
import os, json
os.makedirs(CORPUS_DIR, exist_ok=True)
CORPUS = {
    "car_specs.md": "# Aurora Racing · Team Handbook · Section 3 · Car specification (AR-26)\n\nThe AR-26 is the team's 2026 car. Both cars 27 and 88 share the same specification except for the setup items listed in Section 1.\n\n- Power unit: Aurora PU-26, 1.6 litre V6 turbo hybrid, supplied by the team's own engine division.\n- Minimum weight: 798 kg including the driver.\n- Wheelbase: 3,600 mm.\n- Gearbox: eight forward gears plus reverse, seamless shift.\n- Brakes: carbon discs, 328 mm front and 280 mm rear.\n- Fuel tank capacity: 110 kg.\n- Steering wheel: 22 buttons and 6 rotaries, including the fuel rotary and the overtake button.\n- Data: about 300 sensors, streaming roughly 1.5 GB per race weekend per car to the pit wall and to the factory in Silverstone.\n",
    "comms_protocol.md": "# Aurora Racing · Team Handbook · Section 8 · Communications protocol\n\n## 8.1 Who talks to the driver\n\nDuring a stint only the race engineer speaks to the driver on the driver channel. The strategist, the chief mechanic and the team principal speak to the race engineer on the intercom, and the race engineer decides what to pass on. The only exception is a safety call, which anyone on the pit wall may make directly.\n\n## 8.2 Channels\n\nThere are three radio channels. The driver channel connects the driver and the race engineer. The crew channel connects the pit-stop crew and the chief mechanic; the \"check\" calls of the wet pit stop are made here. The intercom connects everyone on the pit wall and the factory support room.\n\n## 8.3 Timing\n\nMessages to the driver are sent on the straights, never in a braking zone. Long messages are split. The race engineer waits for \"copy\" before sending another instruction.\n",
    "drivers.md": "# Aurora Racing · Team Handbook · Section 1 · Drivers\n\n## Car 27 · Luca Ferrand\n\nLuca Ferrand drives car 27. He was born in Lyon, France, in 1998 and joined Aurora Racing in 2022 after two seasons in Formula 2, where he finished third in the 2021 championship. His race engineer is Priya Nandakumar. Ferrand prefers a front-limited car and tends to ask for more front wing during a stint. His radio style is short; he rarely speaks unless asked.\n\n## Car 88 · Mika Sørensen\n\nMika Sørensen drives car 88. She was born in Aarhus, Denmark, in 2001 and joined the team in 2024 from the Aurora junior programme. Her race engineer is Tomás Reyes. Sørensen runs a stiffer rear anti-roll bar than Ferrand, which is why car 88 uses a slightly higher rear tyre pressure in the wet. She talks more on the radio and gives detailed feedback about tyre behaviour, which the strategists value.\n\n## Reserve driver\n\nThe reserve driver is Kenji Watanabe, who also runs the simulator programme on Friday evenings. He has no race starts with the team.\n",
    "fuel_and_energy.md": "# Aurora Racing · Team Handbook · Section 5 · Fuel and energy\n\n## 5.1 Fuel load\n\nThe race fuel load is the calculated consumption for the race distance plus a safety margin. The standard margin is 0.8 kg. The margin is set on Saturday evening by the performance engineer and cannot be changed after the car leaves the garage for the formation lap.\n\n## 5.2 Fuel rule under safety car\n\nWhen the safety car is deployed, consumption drops but the race can be extended by a restart at full pace. The rule is that under safety car the driver switches to fuel mode 3 and the target margin is raised to 1.5 kg. The race engineer confirms the new margin on the radio with the phrase \"margin one five\". If the margin cannot be recovered before the restart, the driver is asked to lift and coast for the rest of the race.\n\n## 5.3 Energy deployment\n\nThe energy store may deploy up to 4 MJ per lap. The default deployment map keeps 0.5 MJ in reserve for the overtake button. In \"mode push\" the reserve is released and the full 4 MJ is available for one lap, after which the map returns to default automatically.\n",
    "garage_and_pitlane.md": "# Aurora Racing · Team Handbook · Section 10 · Garage and pit lane\n\nThe garage has two bays, car 27 on the left and car 88 on the right when seen from the pit lane. The tyre area is at the back of the garage and the blankets are powered from the left wall.\n\nThe pit lane speed limit is 80 km/h at most circuits and 60 km/h at circuits with a short pit lane such as Monaco and Singapore. The speed limiter is engaged by the driver at the pit entry line. Exceeding the limit is a penalty for the team.\n\nThe pit box is marked with two lines. The front jack operator stands on the front line and the car must stop with the front wing over it. If the car overshoots, the crew pulls it back before starting the stop.\n",
    "handbook_procedures.md": "# Aurora Racing · Team Handbook · Section 4 · Pit-stop procedures\n\nThis section describes the standard pit-stop procedures for cars 27 and 88. Every procedure has a code in the procedure register (Section 9) so that it can be named on the radio without ambiguity. The chief mechanic owns this section, reviews it after every race weekend and publishes changes on the Monday briefing. Crew members are expected to know the procedure in force before the car leaves the garage for the first time in a session. Practice stops are run on Thursday afternoon and on Saturday morning, and the stationary times are logged in the pit-stop sheet together with the names of the crew on each corner.\n\n## 4.1 Dry-race pit stop (code P-D1)\n\nThe dry pit stop is the default procedure and the one the crew practises most. The crew is 18 people: four wheel-gun operators (one per corner), eight tyre carriers (two per corner, one to remove the used tyre and one to fit the new one), two jack operators (front and rear), two stabilisers who hold the car steady at the sidepods, one front-wing adjuster and one lollipop controller who gives the release signal from the front of the car.\n\nTarget stationary time is 2.4 seconds. The record for the team is 2.1 seconds, set in 2025. Dry tyre pressures at the stop are 22.0 psi front and 20.0 psi rear for car 27, and 22.5 psi front and 20.0 psi rear for car 88. Pressures are set in the garage before the tyres go to the pit wall and are not checked again during the stop; the gauge stays in the garage. The car is released as soon as the four wheel guns confirm with the green light on the gantry and both jacks drop. The front-wing adjuster works only when the race engineer has called an adjustment before the car enters the pit lane, and the adjustment is expressed in turns of the flap screw, for example \"front wing plus two\".\n\nIf a wheel gun fails to confirm, the lollipop controller holds the car and the corner is redone. A stop with a redo is reported to the chief mechanic immediately after the race, together with the video from the garage camera.\n\nCrew positions are fixed for the season and printed on the pit-stop sheet. A crew member who is unavailable for a race is replaced by the named reserve for that corner, and the reserve runs at least ten practice stops on Thursday before being allowed on the live stop. The chief mechanic may swap corners between the two cars only on Saturday morning, never on race day. Every practice and live stop is filmed from the gantry camera, and the video is reviewed on Monday with the stationary times, the release time and the time lost in the pit lane.\n\n## 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88)\n\nThe wet pit stop adds a pressure check to the dry procedure because the intermediate and full wet tyres are more sensitive to pressure and because the blankets run cooler for those compounds. For car 27 in wet conditions the front tyres are set to 23.5 psi and the rear tyres to 21.0 psi. For car 88 the front tyres are set to 23.5 psi and the rear tyres to 21.5 psi, because car 88 runs a stiffer rear anti-roll bar and needs a little more support from the tyre.\n\nBefore the car is released, one of the tyre carriers on each side reads the pressure on the gauge and calls \"check\" on the crew channel. The lollipop controller may only release the car after hearing four \"check\" calls, one per corner. This adds about 0.8 seconds to the stationary time and is accepted; a wet stop with a stationary time of 3.2 seconds is considered good. There is no warm-up sequence in the wet procedure; the tyres come from the blankets directly to the car and the blankets for intermediates and wets are set to 40 degrees Celsius as described in Section 2.\n\nVisibility in the pit box is worse in the wet, so the stabilisers wear the high-visibility vests and the lollipop controller uses the lit paddle instead of the standard board. Wheel-gun operators kneel on the anti-slip mats, which are laid out as soon as rain is forecast for the session.\n\n## 4.3 Pit stop under safety car (code P-SC)\n\nUnder the safety car the pit lane is usually crowded, because most of the field stops at the same time to take advantage of the reduced time loss. The crew leaves the garage only after the strategist confirms the gap to the car behind and the expected position at the pit exit. The procedure is otherwise identical to the dry or wet procedure in force at the time: pressures, crew positions and release rules do not change.\n\nTwo details are specific to the safety-car stop. The front-wing adjuster does not touch the wing unless the race engineer calls the adjustment explicitly, because the time gained by the adjustment rarely justifies the risk of an unsafe release in a crowded lane. And the lollipop controller checks the pit lane for approaching cars before releasing, in addition to the wheel-gun lights, since an unsafe release under safety car is the most common penalty in this scenario.\n\n## 4.4 Red flag (code P-RF)\n\nUnder a red flag both cars return to the garage and the session is suspended. Tyres may be changed in the garage without the pit-stop crew, using the garage jacks and the standard wheel guns. The pressure values in force are the same as in 4.1 or 4.2 depending on the conditions declared by race control. The chief mechanic decides whether the car goes back to the grid on the same set or on a new set, after consulting the strategist about the remaining race distance and the tyre allocation.\n\nDuring the suspension the crew may also repair minor damage, replace the front wing and adjust the fuel load only if race control allows it; the race engineer confirms what is allowed on the intercom before any work starts. The car leaves the garage for the restart only when the chief mechanic gives the go on the crew channel.\n",
    "handbook_tyres.md": "# Aurora Racing · Team Handbook · Section 2 · Tyres\n\n## 2.1 Compounds\n\nFive dry compounds are available to the team during a season, named C1 to C5. C1 is the hardest compound and C5 is the softest. The three compounds nominated for a given race weekend are called hard, medium and soft, so the same physical compound can be the \"soft\" at one race and the \"medium\" at another. Intermediate tyres are marked with a green band and full wet tyres with a blue band.\n\n## 2.2 Reference pressures\n\nThe reference pressures below are the garage settings before the tyres go to the pit wall. Race-specific values are published on the Friday tyre sheet and override this table.\n\n| Condition | Front (psi) | Rear (psi) |\n|---|---|---|\n| Dry, car 27 | 22.0 | 20.0 |\n| Dry, car 88 | 22.5 | 20.0 |\n| Wet, car 27 | 23.5 | 21.0 |\n| Wet, car 88 | 23.5 | 21.5 |\n\n## 2.3 Degradation\n\nThe medium compound is expected to lose about 0.08 seconds per lap of pace from thermal degradation on a typical circuit. The soft loses roughly twice that. When the lap-time loss exceeds 0.4 seconds relative to the first lap of the stint, the tyre is considered past its useful window and the strategist evaluates a stop.\n\n## 2.4 Blankets\n\nTyre blankets are set to 70 degrees Celsius for dry compounds and 40 degrees for intermediates and wets. Blankets stay on until the tyre carriers pick the set up for the stop.\n",
    "logistics.md": "# Aurora Racing · Team Handbook · Section 11 · Logistics\n\nThe team travels with two race cars, one spare chassis and about 45 tonnes of freight for European rounds. For fly-away rounds the freight is split into sea freight, which leaves six weeks ahead, and air freight, which leaves on the Sunday before the race. The garage is built on Tuesday and Wednesday and dismantled on Sunday night after the race. The travelling team is about 60 people, and the factory in Silverstone supports every session from the operations room.\n",
    "procedure_codes.md": "# Aurora Racing · Team Handbook · Section 9 · Procedure register\n\nEvery operational procedure has a short code so it can be named on the radio without ambiguity.\n\n| Code | Procedure | Applies to |\n|---|---|---|\n| P-D1 | Dry-race pit stop | Both cars |\n| P27-W1 | Wet-race pit stop, car 27 | Car 27 |\n| P88-W1 | Wet-race pit stop, car 88 | Car 88 |\n| P-SC | Pit stop under safety car | Both cars |\n| P-RF | Red-flag garage procedure | Both cars |\n| Q-1 | Qualifying out-lap and push-lap sequence | Both cars |\n| G-3 | Garage evacuation | Everyone |\n\nWhen a procedure changes, the chief mechanic increments the number at the end of the code. P27-W1 is the first version of the car 27 wet pit stop; a revised version would be P27-W2.\n",
    "radio_glossary.md": "# Aurora Racing · Team Handbook · Section 7 · Radio glossary\n\nRadio time is scarce and every phrase has one meaning. Drivers and engineers use the phrases below exactly as written.\n\n- **\"Box, box.\"** Pit this lap. Said twice so it is never confused with anything else. The driver confirms with \"box\".\n- **\"Stay out.\"** Cancel a planned stop; the driver keeps driving.\n- **\"Push now.\"** Drive at maximum pace for the next laps, usually to build a gap before a stop.\n- **\"Lift and coast.\"** Lift the throttle earlier before braking zones to save fuel. The engineer may add a number of metres, for example \"lift and coast 50\".\n- **\"Fuel mode 3.\"** Switch the steering-wheel fuel rotary to position 3, the most fuel-saving map.\n- **\"Copy.\"** The message was understood.\n- **\"Mode push.\"** Engine map for maximum power, used for overtakes and qualifying laps.\n- **\"Overtake button.\"** The button on the steering wheel that unlocks the extra deployment for a few seconds.\n- **\"Plan A / Plan B.\"** The two strategies discussed before the race. Plan B is usually the one-stop alternative.\n- **\"Gap ahead / gap behind.\"** The time to the car in front or behind, in seconds.\n",
    "sponsor_obligations.md": "# Aurora Racing · Team Handbook · Section 12 · Sponsor obligations\n\nEach driver has four appearance slots per race weekend, usually on Thursday. Media sessions are scheduled by the communications team and never overlap with an engineering briefing. Partner logos on the car and overalls follow the branding guide; the position of the title partner logo on the engine cover cannot be changed without approval. Drivers wear the team cap in every interview.\n",
    "strategy_playbook.md": "# Aurora Racing · Team Handbook · Section 6 · Strategy playbook\n\n## 6.1 Undercut\n\nAn undercut is stopping before the car you are racing so that fresh tyres gain you the position when the other car stops later. The playbook recommends the undercut when the tyre degradation of the car ahead is visible (its lap times are dropping by more than 0.3 seconds per lap) and the pit lane is clear. The undercut is not attempted if the driver would rejoin in traffic slower than the current pace.\n\n## 6.2 Overcut\n\nAn overcut is staying out longer than the rival. It works when the rival rejoins in traffic or when the track is improving faster than the tyres degrade, which is common in drying conditions.\n\n## 6.3 Plan A and Plan B\n\nPlan A is discussed on Saturday and is usually a two-stop race. Plan B is the one-stop alternative used when a safety car or a slower degradation makes it possible. The switch between plans is called by the head of strategy and communicated to the driver with the phrase \"Plan B\".\n\n## 6.4 Wet-to-dry transition\n\nWhen the track dries, the strategist watches sector times of the cars on slicks. The cross-over point is when a car on slicks matches the intermediate lap time. The car that stops first at the cross-over usually gains a position.\n",
}
for name, text in CORPUS.items():
    open(os.path.join(CORPUS_DIR, name), 'w', encoding='utf-8').write(text)
print(len(CORPUS), 'documentos gravados em', CORPUS_DIR)


12 documentos gravados em corpus
⏱ 0.0 s


In [7]:
# Célula 0 · Preparação (rode ANTES da aula). Constrói o índice de 512 tokens e aquece os modelos.
docs  = load_corpus()
rows  = make_chunks(docs, chunk_size=512, overlap=51)
index = build_index(rows, "handbook_512")
print(f"{len(docs)} documentos, {len(rows)} chunks, {index.count()} vetores no índice · modelo {EMB_MODEL}")
assert sum(count_tokens(d["text"]) for d in docs) > 2000, "corpus incompleto: rode a célula anterior"
warm_up()
print("Ollama:", requests.get(f"{OLLAMA_URL}/api/version").json())

12 documentos, 14 chunks, 14 vetores no índice · modelo nomic-embed-text
qwen2.5:3b               carregado em GPU
nomic-embed-text:latest  carregado em GPU
Ollama: {'version': '0.34.0'}
⏱ 136.2 s


---
## ▶ LIVE 1 · A mesma pergunta, sem documentos

*Slide 4.* O modelo responde à pergunta do carro 27 sem nenhum contexto. Leia em voz alta e pergunte à turma de onde vieram os números.

In [8]:
QUESTION = "What is the pit-stop tyre pressure protocol for car 27 in wet conditions?"

answer, secs = generate(build_prompt(QUESTION, hits=[], grounded=False))
print(answer)
print(f"\n({GEN_MODEL}, sem contexto, {secs:.1f} s)")

I'm sorry, but I don't have specific information about the pit-stop tyre pressure protocol for car 27 in wet conditions. Pit-stop tyre pressure protocols can vary significantly based on the team, the race, and the specific conditions. To provide an accurate answer, I would need more detailed information about the race and the team's strategy. If you have access to the latest race reports or the team's official strategy briefings, you might find the specific details you're looking for.

(qwen2.5:3b, sem contexto, 11.8 s)
⏱ 11.8 s


---
## ▶ LIVE 2 · Uma página do manual, três tamanhos de chunk

*Slide 14.* A página de procedimentos fatiada em 128, 512 e 2048 tokens. Para cada tamanho, quantos chunks saíram e em qual deles caiu a linha da pressão dos pneus.

In [9]:
page = next(d for d in docs if d["doc"] == "handbook_procedures.md")
print(f"handbook_procedures.md tem {count_tokens(page['text'])} tokens\n")

# a seção 4.2 inteira, para medir quanto dela viaja junto com a linha da pressão
sec42 = next(s for s in page["text"].split("## ") if s.startswith("4.2"))
sentences42 = [x.strip() for x in re.split(r"(?<=[.!?])\s+", sec42) if len(x.strip()) > 20]

def coverage(chunk):                       # fração das frases da seção 4.2 presentes no chunk
    return sum(1 for x in sentences42 if x in chunk) / len(sentences42)

for size in (128, 512, 2048):
    chunks = chunk_fixed(page["text"], chunk_size=size, overlap=size // 10)
    hit = next(i for i, c in enumerate(chunks) if "23.5 psi" in c)
    c = chunks[hit]
    print(f"chunk_size={size:<5} {len(chunks)} chunks · a pressão está no chunk {hit} ({count_tokens(c)} tokens)")
    print(f"    quanto da seção 4.2 (wet) veio junto: {coverage(c):.0%}")
    print(f"    outras seções no mesmo chunk: {[x for x in re.findall(r'^## (4\.\d)', c, re.M) if x != '4.2'] or 'nenhuma'}\n")

handbook_procedures.md tem 1056 tokens

chunk_size=128   9 chunks · a pressão está no chunk 4 (128 tokens)
    quanto da seção 4.2 (wet) veio junto: 33%
    outras seções no mesmo chunk: nenhuma

chunk_size=512   3 chunks · a pressão está no chunk 1 (512 tokens)
    quanto da seção 4.2 (wet) veio junto: 100%
    outras seções no mesmo chunk: ['4.3', '4.4']

chunk_size=2048  1 chunks · a pressão está no chunk 0 (1056 tokens)
    quanto da seção 4.2 (wet) veio junto: 100%
    outras seções no mesmo chunk: ['4.1', '4.3', '4.4']

⏱ 0.0 s


---
## ▶ LIVE 3 · O que o banco devolve para o carro 27

*Slide 24.* A mesma pergunta, k igual a 1, 4 e 12. A turma acabou de apostar o que acontece com k grande. Observe a distância crescer e o assunto mudar a partir do quinto ou sexto resultado.

In [10]:
Q = "What tyre pressure do we use for car 27 in the wet?"

for k in (1, 4, 12):
    print(f"===== top_k = {k} =====")
    show(retrieve(index, Q, k=k))
    print()

===== top_k = 1 =====
[1] handbook_procedures.md · 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88)  (distance 0.292)
    reviewed on Monday with the stationary times, the release time and the time lost in the pit lane.  ## 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 ...

===== top_k = 4 =====
[1] handbook_procedures.md · 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88)  (distance 0.292)
    reviewed on Monday with the stationary times, the release time and the time lost in the pit lane.  ## 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 ...
[2] handbook_tyres.md · 2.2 Reference pressures  (distance 0.327)
    # Aurora Racing · Team Handbook · Section 2 · Tyres  ## 2.1 Compounds  Five dry compounds are available to the team during a season, named C1 to C5. C1 is the h ...
[3] handbook_procedures.md · 4.4 Red flag (code P-RF)  (distance 0.332)
    Tyres may be changed in the garage without the pit-stop crew, using th

---
## ▶ LIVE 4 · Prompt aumentado, duas perguntas

*Slide 29.* A pergunta que está no corpus e a que não está, cada uma com e sem a regra de grounding. A resposta que importa é a quarta.

In [13]:
questions = [
    "What tyre pressure do we use for car 27 in the wet?",          # está no manual
    "How many championship points does car 27 have this season?",   # não está
]
print(f"As duas versões recebem os mesmos {4} chunks recuperados. Só a instrução de sistema muda.\n")

for q in questions:
    hits = retrieve(index, q, k=4)
    for grounded in (True, False):
        answer, secs = generate(build_prompt(q, hits, grounded=grounded), max_tokens=200)
        label = "contexto + regra de grounding" if grounded else "contexto sem regra"
        print(f"Q: {q}\n[{label}] ({secs:.1f} s)\n{answer}\n")
        print("Sources used (from retrieval): " + "; ".join(sources_used(hits)) + "\n")
    print("-" * 80)

As duas versões recebem os mesmos 4 chunks recuperados. Só a instrução de sistema muda.

Q: What tyre pressure do we use for car 27 in the wet?
[contexto + regra de grounding] (2.1 s)
For car 27 in wet conditions, the front tyres are set to 23.5 psi and the rear tyres to 21.0 psi. [handbook_procedures.md · 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88)]

Sources used (from retrieval): handbook_procedures.md · 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88); handbook_tyres.md · 2.2 Reference pressures; handbook_procedures.md · 4.4 Red flag (code P-RF); handbook_procedures.md · 4.1 Dry-race pit stop (code P-D1)

Q: What tyre pressure do we use for car 27 in the wet?
[contexto sem regra] (1.4 s)
For car 27 in wet conditions, the front tyres are set to 23.5 psi and the rear tyres to 21.0 psi.

Sources used (from retrieval): handbook_procedures.md · 4.2 Wet-race pit stop (code P27-W1 for car 27, code P88-W1 for car 88); handbook_tyres.md · 2.2 

---
## ⏱ Tempos desta sessão

Rode ao fim do ensaio para planejar a aula. A célula 0 não conta para a aula (roda antes).

In [ ]:
# Resumo dos tempos de todas as células rodadas nesta sessão.
import pandas as pd
times = pd.DataFrame(CELL_TIMES)
print(f"total: {times['seconds'].sum():.1f} s em {len(times)} células")
times
